# 02a — Multi-CW Coordinate-Ascent Experiment Runner

Runs deterministic pulsar-distance recovery experiments using `02_multicw_pairwise_coordinate_ascent.py`.

Key rule: truth is used only for scoring. During optimization, all non-scanned distances are current chain guesses.

Default cells run small sanity jobs. Increase `N_PSR`, `N_CHAINS`, `MAX_SWEEPS`, and remove `MAX_PAIRS_PER_SWEEP` for science sweeps.


In [ ]:
from pathlib import Path
import itertools
import subprocess
import shlex
import json
import time

HERE = Path.cwd()
DRIVER = HERE / "02_multicw_pairwise_coordinate_ascent.py"
OUTDIR = HERE / "02_multicw_pairwise_outputs"
OUTDIR.mkdir(exist_ok=True)
print(DRIVER)
print(OUTDIR)


## Sweep Configuration

Suggested progression:

1. `N_PSR=10`, `MAX_PAIRS_PER_SWEEP=8` to catch failures quickly.
2. `N_PSR=20-40`, `anchor_ladder`, full pair schedule for method development.
3. `N_PSR=80`, `anchor_ladder`, few chains for production timing.
4. `pair_mode='all'` only for small `N_PSR`; all pairs scales as `N(N-1)/2`.


In [ ]:
# Small default. Edit for larger runs.
# DATA_MODE="stochastic" uses the sandbox-style enterprise injection:
# white noise + intrinsic red noise + optional HD GWB, then matching discovery GP likelihood.
# DATA_MODE="pure" keeps the fast/noise-free CW-only path for optimizer debugging.
DATA_MODE = "stochastic"
STOCHASTIC_SCENARIO = "well_separated"
COMPONENTS = 10
GWB_LOG10_A = -17.5
GWB_GAMMA = 13/3
INCLUDE_GWB = True

N_PSR = 12
N_CHAINS = 3
MAX_SWEEPS = 2
PAIR_MODE = "anchor_ladder"
MAX_PAIRS_PER_SWEEP = 10  # 0 = full chosen schedule
MIN_POINTS = 21
POINTS_PER_MODE = 4
PROGRESS_EVERY = 10

N_CW_GRID = [2, 3, 4]
LOG10_H_GRID = [-13.0, -12.5, -12.0]
NOISE_GRID = [
    {"label": "matching_gp", "extra_white_rms": 0.0, "extra_red_rms": 0.0, "extra_common_red_rms": 0.0},
]

RUN_SWEEP = False  # set True to launch jobs from notebook


In [ ]:
def build_cmd(n_cw, log10_h, noise_cfg, seed_offset=0):
    cmd = [
        "rtk", "python", str(DRIVER.name),
        "--data-mode", DATA_MODE,
        "--n-psr", str(N_PSR),
        "--n-cw", str(n_cw),
        "--n-chains", str(N_CHAINS),
        "--max-sweeps", str(MAX_SWEEPS),
        "--pair-mode", PAIR_MODE,
        "--max-pairs-per-sweep", str(MAX_PAIRS_PER_SWEEP),
        "--min-points", str(MIN_POINTS),
        "--points-per-mode", str(POINTS_PER_MODE),
        "--progress-every", str(PROGRESS_EVERY),
        "--log10-h", str(log10_h),
        "--seed", str(12345 + seed_offset),
        "--noise-seed", str(24680 + seed_offset),
    ]
    if DATA_MODE == "stochastic":
        cmd += [
            "--stochastic-scenario", STOCHASTIC_SCENARIO,
            "--components", str(COMPONENTS),
            "--gwb-log10-a", str(GWB_LOG10_A),
            "--gwb-gamma", str(GWB_GAMMA),
        ]
        if not INCLUDE_GWB:
            cmd.append("--no-gwb")
    else:
        # Optional residual-domain stress knobs for pure-mode robustness tests.
        cmd += [
            "--extra-white-rms", str(noise_cfg["extra_white_rms"]),
            "--extra-red-rms", str(noise_cfg["extra_red_rms"]),
            "--extra-common-red-rms", str(noise_cfg["extra_common_red_rms"]),
        ]
    return cmd

jobs = []
for idx, (n_cw, log10_h, noise_cfg) in enumerate(itertools.product(N_CW_GRID, LOG10_H_GRID, NOISE_GRID)):
    jobs.append((n_cw, log10_h, noise_cfg, build_cmd(n_cw, log10_h, noise_cfg, idx)))

for n_cw, log10_h, noise_cfg, cmd in jobs:
    print(f"n_cw={n_cw} log10_h={log10_h} mode={DATA_MODE} noise={noise_cfg['label']}")
    print(shlex.join(cmd))


In [ ]:
if RUN_SWEEP:
    for idx, (n_cw, log10_h, noise_cfg, cmd) in enumerate(jobs, 1):
        print("="*100)
        print(f"job {idx}/{len(jobs)}  n_cw={n_cw} log10_h={log10_h} noise={noise_cfg['label']}")
        print(shlex.join(cmd))
        t0 = time.time()
        proc = subprocess.run(cmd, cwd=HERE, text=True, capture_output=True)
        print(proc.stdout[-5000:])
        if proc.returncode != 0:
            print(proc.stderr[-5000:])
            raise RuntimeError(f"job failed with returncode {proc.returncode}")
        print(f"seconds={time.time()-t0:.1f}")
else:
    print("RUN_SWEEP=False; no jobs launched.")


## Pure-Mode Residual Stress Sweep

Main science path now uses `DATA_MODE="stochastic"`: enterprise simulation draws white noise, intrinsic red noise, and optional HD GWB, while discovery evaluates the matching stochastic GP likelihood with injected hyperparameters fixed.

This section is only for fast pure-mode stress tests using residual-domain perturbations. It is useful for optimizer robustness, not for final stochastic-noise claims.


In [ ]:
PURE_STRESS_GRID = [
    {"label": "clean", "extra_white_rms": 0.0, "extra_red_rms": 0.0, "extra_common_red_rms": 0.0},
    {"label": "white_1e-8", "extra_white_rms": 1e-8, "extra_red_rms": 0.0, "extra_common_red_rms": 0.0},
    {"label": "red_1e-8", "extra_white_rms": 0.0, "extra_red_rms": 1e-8, "extra_common_red_rms": 0.0},
    {"label": "common_red_1e-8", "extra_white_rms": 0.0, "extra_red_rms": 0.0, "extra_common_red_rms": 1e-8},
]
RUN_PURE_STRESS_SWEEP = False

old_mode = DATA_MODE
DATA_MODE = "pure"
noise_jobs = []
for idx, noise_cfg in enumerate(PURE_STRESS_GRID):
    noise_jobs.append((noise_cfg, build_cmd(n_cw=3, log10_h=-12.0, noise_cfg=noise_cfg, seed_offset=1000+idx)))
DATA_MODE = old_mode

for cfg, cmd in noise_jobs:
    print(cfg['label'], shlex.join(cmd))

if RUN_PURE_STRESS_SWEEP:
    for cfg, cmd in noise_jobs:
        print("="*100)
        print(cfg)
        proc = subprocess.run(cmd, cwd=HERE, text=True, capture_output=True)
        print(proc.stdout[-5000:])
        if proc.returncode != 0:
            print(proc.stderr[-5000:])
            raise RuntimeError(f"job failed: {cfg['label']}")
else:
    print("RUN_PURE_STRESS_SWEEP=False; no pure stress jobs launched.")
